# FlowMind on Colab

Runs the FlowMind pipeline on a Colab runtime, with Google Drive holding everything
the repo gitignores (dataset, fitted models, trace logs, model weights) so it survives
runtime disconnects.

## Two ways to run this

**A. VS Code + the official Colab extension (recommended).** Install the `Colab`
extension (Google, first-party, on the VS Marketplace and Open VSX), open this file
from your local clone, then `Select Kernel → Colab` and pick a GPU runtime. You keep
this notebook and the whole repo on local disk under normal git, and cells execute on
a Colab T4.

**B. Browser.** [colab.research.google.com](https://colab.research.google.com) →
`File → Upload notebook`, then `Runtime → Change runtime type → T4 GPU`.

⚠️ **Either way, cells run against the *remote* filesystem.** The extension syncs
nothing — a relative path like `data/train_full.json` resolves on the Colab VM, not
your Mac. That's why §0 still clones the repo and mounts Drive: the code and data have
to physically be on the runtime. Consequence worth knowing: editing `flowmind/*.py`
locally does **not** affect a run until you push and let §0.3 pull it.

**Runtime choice — this matters for your GPU quota:**

| Sections | Runtime | Why |
|---|---|---|
| §0–§4 (setup, tests, M1 metric, classifier) | **CPU** | The deterministic lane is pure graph algorithms + TF-IDF. Half a second on CPU. A GPU here is wasted quota. |
| §5 (Qwen3-VL Reader) | **T4 GPU** | Only the VLM needs it. |

Spec reference: `flowmind_feature_spec.md`. Milestones: §8 of the README.

## §0 Setup

### 0.1 Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 0.2 Paths

The repo itself lives on the ephemeral VM disk (it's 400 KB and on GitHub — cheap to
re-clone). Only the heavy, gitignored artifacts go to Drive:

```
MyDrive/FlowMind/
├── data/train_full.json      # upload once
├── data/test_full.json       # upload once  (train_router.py needs this)
├── data/images/main/*.png    # 2532 PNGs
├── data/images/bottom_top/*.png
├── models/                   # question_classifier.joblib
├── runs/                     # trace JSONL (spec §12)
└── hf_cache/                 # Qwen3-VL weights, ~5GB
```

In [ ]:
import os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/FlowMind')   # change if you want it elsewhere
REPO  = Path('/content/FlowMind')
REPO_URL = 'https://github.com/PurvajaNarayan/FlowMind.git'

for sub in ('data/images/main', 'data/images/bottom_top', 'models', 'runs', 'hf_cache'):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

print('Drive workspace ready:', DRIVE)

### 0.3 Clone (or update) the repo

Re-running this cell on a later session pulls instead of failing.

In [ ]:
if REPO.exists():
    !cd {REPO} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO}

%cd {REPO}
!git log --oneline -3

### 0.4 Point the repo's gitignored paths at Drive

Symlinks, so the code and tools work with their default relative paths
(`data/train_full.json`, `models/…`, `runs/…`) and nothing needs a `--flag`.

Note we link the *files* inside `data/`, not `data/` itself — `data/README.md` is
tracked, and replacing the whole directory would show up as a deletion in
`git status`. Every path linked below is gitignored, so the working tree stays clean.

In [ ]:
LINKS = {
    'data/train_full.json': DRIVE / 'data/train_full.json',
    'data/test_full.json':  DRIVE / 'data/test_full.json',
    'data/images':          DRIVE / 'data/images',
    'models':               DRIVE / 'models',
    'runs':                 DRIVE / 'runs',
}

import shutil

for rel, target in LINKS.items():
    link = REPO / rel
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    elif link.exists():
        link.unlink()
    link.symlink_to(target)
    print(f'{rel:24} -> {target}')

# .gitignore writes these as `data/images/`, `models/`, `runs/` — trailing slashes
# match directories, and a symlink is not a directory, so git would list them as
# untracked. .git/info/exclude is per-clone and untracked, so we fix it here rather
# than editing the shared .gitignore.
exclude = REPO / '.git/info/exclude'
existing = exclude.read_text() if exclude.exists() else ''
needed = [r for r in LINKS if r not in existing]
if needed:
    with exclude.open('a') as fh:
        fh.write('\n# Colab: Drive symlinks (see notebooks/flowmind_colab.ipynb)\n')
        fh.write('\n'.join(needed) + '\n')

print('\ngit status (should be clean):')
!git status --short

### 0.5 Install core deps

`networkx`, `scikit-learn`, `joblib`, `pytest` — Colab preinstalls most of these, so
this is usually a no-op. This is also the step that fixes the numpy/sklearn ABI
mismatch you hit locally under anaconda.

In [ ]:
!pip install -q -r requirements.txt
import networkx, sklearn, joblib
print('networkx', networkx.__version__, '| sklearn', sklearn.__version__)

In [ ]:
# --- 0.5b One-time: stage the dataset into Drive from the uploaded zip ---
# Build the zip locally with `python tools/make_colab_subset.py`, upload it to
# MyDrive/FlowMind/, then run this cell once. Later sessions detect the staged
# data and skip straight past.
import subprocess

ZIP = DRIVE / 'flowvqa_colab_subset.zip'
staged = len(list((DRIVE / 'data/images/main').glob('*.png')))

if staged:
    print(f'{staged} images already staged in Drive - nothing to do')
elif ZIP.exists():
    subprocess.run(['unzip', '-q', '-o', str(ZIP), '-d', str(DRIVE / 'data')], check=True)
    n = len(list((DRIVE / 'data/images/main').glob('*.png')))
    print(f'unzipped {n} main-layout images -> {DRIVE}/data')
else:
    print(f'Nothing to stage. Upload flowvqa_colab_subset.zip to {DRIVE}/ , or copy\n'
          f'the dataset into {DRIVE}/data/ manually (layout is in section 0.2).')

### 0.6 Preflight — what's actually staged on Drive?

Tells you which sections can run before you hit a confusing traceback three cells later.

In [ ]:
import json

def check(label, path, need):
    p = Path(path)
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISSING'}  {label:26} {p}")
    if not ok:
        print(f"          -> needed for: {need}")
    return ok

has_train = check('train_full.json', 'data/train_full.json', '§2 M1 metric, §3 classifier')
has_test  = check('test_full.json',  'data/test_full.json',  '§3 classifier (evaluation split)')

for layout in ('main', 'bottom_top'):
    n = len(list((DRIVE / 'data/images' / layout).glob('*.png')))
    print(f"{'OK ' if n else 'MISSING'}  images/{layout:18} {n} PNGs" + ('' if n else '  -> needed for: §5 VLM'))

if has_train:
    ds = json.load(open('data/train_full.json'))
    subsets = {}
    for k in ds:
        subsets[k.rstrip('0123456789')] = subsets.get(k.rstrip('0123456789'), 0) + 1
    nq = sum(len(r.get('qa', {})) for r in ds.values())
    print(f'\ntrain_full.json: {len(ds)} records {subsets}, {nq} QA pairs')

> **If something is MISSING:** upload it into the matching `MyDrive/FlowMind/…` folder
> (drag-and-drop in the Drive web UI is fine for the JSON; use the Drive desktop client
> or a zip + `!unzip` for the 2532 images). You only ever do this once.

## §1 Tests (CPU)

Expect **15 passed** here. Locally you get 14 passed + 1 skipped, because the
anaconda sklearn can't import — on Colab the question-classifier test actually runs.

In [ ]:
!python -m pytest -q -rs

## §2 M1 — the deterministic topological lane (CPU)

The project's hard accuracy number (spec §8): Mermaid parser + graph tool vs FlowVQA's
own topological gold answers. README currently reports **98.8%** overall.

Needs `data/train_full.json`. No GPU, no LLM, no API key.

In [ ]:
!python tools/parser_coverage.py data/train_full.json

Inspect the residual for one subtype — `edge_count` is the weakest lane at 97.1%,
and duplicate-label resolution is the next open M1 task:

In [ ]:
!python tools/parser_coverage.py data/train_full.json --show-fails edge_count

## §3 Question-type classifier (CPU)

TF-IDF + LogReg, no LLM — the router's fallback for unlabeled questions. Trains on
train, evaluates on test, writes `models/question_classifier.joblib` (→ Drive, so the
router picks it up automatically in later sessions instead of falling back to keywords).

**Needs both splits.** If `test_full.json` was MISSING above, this cell will fail.

In [ ]:
!python tools/train_router.py

In [ ]:
# Confirm it persisted to Drive and reloads
!ls -lh models/
from flowmind.question_classifier import QuestionTypeClassifier
clf = QuestionTypeClassifier.load()
for q in ['How many edges exist in the given flowchart?',
          'What happens if the balance check fails?']:
    print(f'{clf.predict(q):18} <- {q}')

## §4 End-to-end smoke test (CPU)

Reader → router → graph tool on a single real record, exercising the frozen `FlowGraph`
contract that every workstream shares.

In [ ]:
from flowmind.data import load_dataset, iter_qa
from flowmind.reader.mermaid_reader import mermaid_to_graph
from flowmind.router import route
from flowmind import graph_tool as gt

ds = load_dataset('data/train_full.json')
item = next(it for it in iter_qa(ds) if it.qa_type == 'topological')
g = mermaid_to_graph(item.mermaid)

print(f'{item.sample_key} q{item.question_id} [{item.subset}]')
print(f'  Q: {item.question}')
print(f'  gold: {item.answers}')
print(f'  intent (from label): {route(item.question, qa_type=item.qa_type)}')
print(f'  intent (free-form):  {route(item.question)}')
print(f'  graph: {gt.node_count(g)} nodes, {gt.edge_count(g)} edges, '
      f'max in-degree {gt.max_indegree(g)}, source={g.source!r}')
print(f'  image: {item.image_path}')

Write a trace to Drive so Owner C's error analysis has something to read (spec §12):

In [ ]:
from flowmind.tracing import Trace, TraceWriter, read_traces

with TraceWriter('runs/colab_smoke.jsonl') as tw:
    tw.write(Trace(sample_key=item.sample_key, question_id=item.question_id,
                   intent='topological', branch='graph_tool',
                   prediction=gt.node_count(g), gold=item.answers[0],
                   correct=str(gt.node_count(g)) == item.answers[0].strip()))

print(read_traces('runs/colab_smoke.jsonl')[-1])

---
## §5 Qwen3-VL Reader — **needs a GPU runtime**

The stretch goal (spec §7.1 / §9): flowchart **image** → Mermaid → `FlowGraph`, reusing
the already-tested parser so the VLM is a drop-in Reader backend.

Before running: `Runtime → Change runtime type → T4 GPU`, then re-run §0 (the VM disk
resets, but Drive and the HF weight cache persist).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch the runtime type before continuing'

### 5.1 Heavy deps + weight cache on Drive

`HF_HOME` points at Drive so the ~5 GB download happens **once** across all future
sessions. Trade-off: loading from Drive's FUSE mount is slower than local disk. If model
load feels sluggish, copy the snapshot to `/content` first and point `HF_HOME` there for
that session.

`requirements-vlm.txt` pins `transformers>=4.57` for Qwen3-VL support. If Colab ships an
older build, pip upgrades it — and you may need **Runtime → Restart session** once
afterwards (then re-run §0, skipping the installs).

In [ ]:
import os
os.environ['HF_HOME'] = str(DRIVE / 'hf_cache')

!pip install -q -r requirements-vlm.txt

import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, '(need >= 4.57)')

### 5.2 Download the weights

Default is `Qwen/Qwen3-VL-2B-Instruct` (`DEFAULT_MODEL_ID` in `vlm_reader.py`). If that
repo id doesn't resolve or is gated, override it — the env var is read by
`QwenVLExtractor`, so nothing in the codebase needs editing:

```python
os.environ['FLOWMIND_VLM_MODEL'] = 'Qwen/Qwen2.5-VL-3B-Instruct'
```

For a gated repo, authenticate first with `huggingface_hub.login()`.

In [ ]:
!python tools/download_vlm.py

### 5.3 One image, end to end

Sanity-check a single flowchart before spending GPU time on a sweep. Compares the VLM's
Mermaid against the dataset's ground-truth `mermaid` field — free supervision, which is
the whole reason the VLM emits Mermaid instead of a graph directly.

In [ ]:
from flowmind.data import image_path_for, load_dataset
from flowmind.eval.metrics import graph_extraction_accuracy
from flowmind.reader.mermaid_reader import mermaid_to_graph
from flowmind.reader.vlm_reader import QwenVLExtractor
from IPython.display import Image as ShowImage, display

ds = load_dataset('data/train_full.json')   # re-loaded so §5 runs standalone after a restart

key = next(k for k in ds if image_path_for(k, 'data', 'main').exists())
img = image_path_for(key, 'data', 'main')
display(ShowImage(filename=str(img), width=420))

ext = QwenVLExtractor()          # lazy load; first call pays the model-load cost
pred_mermaid = ext.image_to_mermaid(str(img))
print('--- VLM output ---');  print(pred_mermaid)
print('--- gold ---');        print(ds[key]['mermaid'])

pred_graph = mermaid_to_graph(pred_mermaid)
print('--- extraction metrics ---')
print(graph_extraction_accuracy(pred_graph, ds[key]['mermaid']))

### 5.4 Zero-shot extraction sweep

The number that decides whether zero-shot is good enough or a LoRA fine-tune is needed.
Saved output doubles as image→mermaid training pairs for that fine-tune.

In [ ]:
!python tools/eval_vlm.py --n 20 --layout main --save runs/vlm_zeroshot_main.jsonl

The directional-bias robustness ablation from `data/README.md` — same charts rendered
bottom-up. A big gap between these two runs means the model is reading layout position
rather than the graph:

In [ ]:
---
## §6 Session notes

**What persists:** everything under `MyDrive/FlowMind` — dataset, `models/*.joblib`,
`runs/*.jsonl`, and the HF weight cache. A fresh runtime only needs §0 re-run.

**What doesn't:** the repo clone on the VM and any pip installs. Colab also reclaims
idle runtimes, so don't leave a long sweep unattended without `--save`.

**Editing code:** the runtime works from the clone made in §0.3, so changes to
`flowmind/*.py` need a push from your machine and a re-run of that cell to land. Only
this `.ipynb` is genuinely local when using the VS Code extension. In practice that's
fine — you'll iterate on cells far more often than on modules — but it does mean a
"quick fix" to `vlm_reader.py` is push-then-pull, not save-and-rerun.

**Committing:** `origin` and `upstream` both point at `PurvajaNarayan/FlowMind`, a
shared team repo, so use a feature branch (`a/label-resolution`, etc.) rather than
`main`, per the README's conventions. With the VS Code extension you just commit
locally as normal. From the browser you'd need a PAT on the runtime:

```python
!git config user.name  'Your Name'
!git config user.email 'you@example.com'
!git checkout -b a/my-feature
# then: git remote set-url origin https://<PAT>@github.com/PurvajaNarayan/FlowMind.git
```

---
## §6 Session notes

**What persists:** everything under `MyDrive/FlowMind` — dataset, `models/*.joblib`,
`runs/*.jsonl`, and the HF weight cache. A fresh runtime only needs §0 re-run.

**What doesn't:** the cloned repo and any pip installs. Colab also reclaims idle
runtimes, so don't leave a long sweep unattended without `--save`.

**Committing from Colab:** `origin` and `upstream` both point at
`PurvajaNarayan/FlowMind` — a shared team repo. Push needs a PAT, and per the README's
conventions use a feature branch (`a/label-resolution`, etc.) rather than `main`:

```python
!git config user.name  'Ayush Sharma'
!git config user.email 'sharma.ayush1@northeastern.edu'
!git checkout -b a/my-feature
# then: git remote set-url origin https://<PAT>@github.com/PurvajaNarayan/FlowMind.git
```

Easier for notebook-only work: download the changed files and commit locally.